# Ingest Data with Zerobus Ingest

To run this notebook you need the following:
- Serverless Compute or Classic Compute Cluster with DBR >= 16.4 LTS
- Zerobus Ingest Public Preview enabled

#### Zerobus Ingest
Zerobus Ingest allows to efficiently push data into tables with ease. It supports record-by-record ingestion at any scale and operates in a serverless environment.

<br>

<img src="./images/zerobus-architecture.png" width="600"/>

<br>

- When data is transmitted to the Row Ingestion API, it goes through a buffering process before being added to a Delta table. This creates an efficient and durable ingestion mechanism to support a high volume of clients with variable throughput.
- Once the data has been materialized into Delta format, it becomes fully compatible with the comprehensive Databricks Platform, allowing users to leverage familiar tools and functionalities for further data analysis and processing.
- By default, the Python SDK performs automatic recovery. When a stream fails (e.g., due to a transient network issue), the SDK will attempt to reconnect and re-ingest any unacknowledged records in the background, preserving their order.

##### Install required libraries

In [0]:
# Install Zerobus Ingest SDK (sync client with JSON support)
%pip install databricks-zerobus-ingest-sdk>=0.2.0
%restart_python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 12.2 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Install custom data generator library
%pip install -r ./line_data_generator/requirements.txt ./line_data_generator
%restart_python

Processing ./line_data_generator
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Cloning https://github.com/QuentinAmbard/mandrova to /tmp/pip-req-build-xy8rhm99
  Running command git clone --filter=blob:none --quiet https://github.com/QuentinAmbard/mandrova /tmp/pip-req-build-xy8rhm99
  Resolved https://github.com/QuentinAmbard/mandrova to commit 553986e2ab1e5e349e095b83f7c1a1e1226f99e0
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 75.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 50.7 MB/s eta 0:00:00
  Created wheel for line-data-generator: filename=line_data_generator-0.1.0-py3-none-any.whl size=1040 sha256=251cd11118c8ce591b8969cbe7f64c12725aed989fe307664213de3726568b62
  Stored in directory: /tmp/pip-ephem-wheel-cache-e3efk0p0/wheels/78/a9/4d/ad17ed2e7bfb12372e07c3d6df011ab850a70e987

##### Provide required information to connect to Zerobus host

To use the Zerobus Ingest SDK you will need the following information:

- Databricks Workspace URL
- Your Bronze table definition
- Zerobus Host URL (format: `<workspace_id>.zerobus.<region>.cloud.databricks.com`)
- Service Principal Id and Secret

**Once you have this information, update the 0-Parameters.ipynb notebook with your own values.**

In [0]:
%run ./0-Parameters

## Parameters
This notebook contains the parameters needed to customise the solution accelerator to you environment. Be sure to modify them before starting to ensure that the accelerator deploys correctly.

#### Notebook 1 & 2: Sensor data generation & Zerobus ingestion  
##### Provide required information

To use the Zerobus Ingest SDK you will need the following information:

- Your table name
  - Identify the target table you want to ingest data to. This is the table created in the notebook "1. Create-Sensor-Bronze-Table"

- Databricks Workspace URL 
  - Get your workspace URL. When viewing your Databricks workspace after logging in, take a look at the URL in your browser with the following format: https://_your_databricks-instance_.com/o=XXXXX. The URL that you require is everything before the “/o=XXXXX”

- Zerobus Host 
  - Zerobus URI = _workspace_id_.ingest.cloud.databricks.com. Databricks will provide you with this URL

- Service Principal --->  _For the following steps you need to be a workspace admin. If you are not an admin, ask to the appropriate person in your organization_
  - Go to Settings > Identity and Access.
  - Generate and save client ID and secret for that Service Principal.

#### Notebook 3: Mapping pipeline 

#### Notebook 4 & 5: Setup Lakebase & front-end app 

### Step 1: Grant permissions to the Service Principal

The Service Principal needs MODIFY and SELECT on the destination table,
plus USE CATALOG and USE SCHEMA.

In [0]:
# Grant MODIFY and SELECT on the table to the Service Principal
spark.sql(f"GRANT MODIFY, SELECT ON TABLE {BRONZE_TABLE} TO `{CLIENT_ID}`")

# Extract catalog and schema from the table name
catalog, schema, _ = BRONZE_TABLE.split(".")

# Grant USE CATALOG on the catalog to the Service Principal
spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `{CLIENT_ID}`")

# Grant USE SCHEMA on the schema to the Service Principal
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `{CLIENT_ID}`")

print(f"Permissions granted on {BRONZE_TABLE} to SP {CLIENT_ID}")

Permissions granted on serverless_simplot_v1_catalog.frozen_potato.potato_sensor_bronze to SP d42a00ff-2702-4fa4-8837-1a8df2ad3069


### Step 2: Setup Zerobus connection (Sync SDK + JSON)

We use the **synchronous** SDK with **JSON record type** for simplicity and reliability.
No protobuf compilation needed — records are sent as plain Python dictionaries.

In [0]:
from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import (
    RecordType,
    StreamConfigurationOptions,
    TableProperties,
)

print(f"Workspace URL:   {WORKSPACE_URL}")
print(f"Zerobus URL:     {ZEROBUS_URL}")
print(f"Target Table:    {BRONZE_TABLE}")
print(f"Service Principal: {CLIENT_ID}")

# Initialize SDK
sdk = ZerobusSdk(ZEROBUS_URL, WORKSPACE_URL)

# Configure for JSON record ingestion
options = StreamConfigurationOptions(record_type=RecordType.JSON)
table_props = TableProperties(BRONZE_TABLE)

print("\n[OK] Zerobus SDK initialized (sync + JSON mode)")

Workspace URL:   https://fe-sandbox-serverless-simplot-v1.cloud.databricks.com
Zerobus URL:     https://7474648424393858.zerobus.us-east-1.cloud.databricks.com
Target Table:    serverless_simplot_v1_catalog.frozen_potato.potato_sensor_bronze
Service Principal: d42a00ff-2702-4fa4-8837-1a8df2ad3069

[OK] Zerobus SDK initialized (sync + JSON mode)


In [0]:
# Test connection
stream = sdk.create_stream(CLIENT_ID, CLIENT_SECRET, table_props, options)
if stream:
    print("[OK] Connection successful — stream created")
    stream.close()
    print("[OK] Test stream closed")
else:
    print("[ERROR] Failed to create stream")

2026-02-25T12:14:55.162129Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=3077f484-1bf7-43a0-9abd-4044aca38e5a
[OK] Connection successful — stream created
2026-02-25T12:14:55.162166Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=3077f484-1bf7-43a0-9abd-4044aca38e5a
2026-02-25T12:14:55.162510Z  INFO databricks_zerobus_ingest_sdk: Successfully created new ephemeral stream stream_id=3077f484-1bf7-43a0-9abd-4044aca38e5a
2026-02-25T12:14:55.162924Z  INFO databricks_zerobus_ingest_sdk: Closing stream stream_id=3077f484-1bf7-43a0-9abd-4044aca38e5a
[OK] Test stream closed


### Step 3: Generate sensor data

We use the frozen potato factory data generator to simulate data from a processing plant:

- **3 production lines** (French Fries, Hash Browns, Wedges)
- Each line has 5-6 sequential processing stages
- Each stage has 1 component (main equipment)
- Each component has 6 sensors:
  - Oil Temperature, Water Temperature, Belt Speed,
  - Freezer Temperature, Moisture Content, Product Weight

In [0]:
import pandas as pd
import time
from line_data_generator import generate_all_lines, generate_equipment_mapping, table_size_estimator

# Frozen Potato Factory configuration
num_lines = 3
machines_per_line = [6, 5, 5]  # stages per line
num_components = 1  # 1 component per stage

In [0]:
# Define sample size (1000 = 16,000 total rows across 16 components)
sample_size = 1000

# Generate equipment mapping
equipment_mapping = generate_equipment_mapping(num_lines, machines_per_line, num_components)

# Estimate table size
tot_num_rows, est_table_size, line_num_rows, est_line_table_size = table_size_estimator(
    machines_per_line, num_components, sample_size
)
print(f"Number of rows: {tot_num_rows}")
print(f"Estimated table size: {est_table_size:.2f} MB")

Number of rows: 16000
Estimated table size: 7.63 MB


In [0]:
# Generate sensor data for all 3 production lines
batch_df_lines = generate_all_lines(equipment_mapping, sample_size, time.time())
display(batch_df_lines.head(10))

Number of rows: 16000


oil_temperature,water_temperature,belt_speed,freezer_temperature,moisture_content,product_weight,component_yield_output,timestamp,component_id,stage_id,damaged_component,abnormal_sensor,line_id
19.506464379342926,14.843580903631967,1.3546625279967035,19.716163338872743,83.28050515623433,496.67539376111455,0.012916420504013277,2026-02-25 13:10:01.808,FF-WASH,Washing,false,None,cd613e30-d8f1-6adf-91b7-584a2265b1f5
20.04953766381992,14.810369164014894,1.3302814214380034,19.761130244064123,81.47948748458703,483.5194408873692,0.023306348351433318,2026-02-25 13:10:01.809,FF-WASH,Washing,false,None,cd613e30-d8f1-6adf-91b7-584a2265b1f5
19.000388953539815,13.24636936077112,1.2484960635997733,20.004808672649194,76.52702721540247,507.4716139820355,0.024698677049403397,2026-02-25 13:10:01.810,FF-WASH,Washing,false,None,cd613e30-d8f1-6adf-91b7-584a2265b1f5
20.185294092202113,16.10784216445175,1.176703642115076,19.908787394486446,76.83932332959338,468.07228129353524,0.02458428613141839,2026-02-25 13:10:01.811,FF-WASH,Washing,false,None,cd613e30-d8f1-6adf-91b7-584a2265b1f5
19.173718819685735,16.768261801860096,1.4304799974696407,19.77401573750889,80.95403182844655,497.17284606794703,0.025702856504915432,2026-02-25 13:10:01.812,FF-WASH,Washing,false,None,cd613e30-d8f1-6adf-91b7-584a2265b1f5
19.90554466687251,15.643353685689842,1.335917782711452,20.34946396423656,79.05494765128361,524.793651910059,0.05561416542949747,2026-02-25 13:10:01.813,FF-WASH,Washing,false,None,cd613e30-d8f1-6adf-91b7-584a2265b1f5
19.996503923993227,16.105000766297522,1.3137570056277155,19.840056136838893,78.61423041490127,514.9290984161146,0.007415601274322511,2026-02-25 13:10:01.814,FF-WASH,Washing,false,None,cd613e30-d8f1-6adf-91b7-584a2265b1f5
19.834417962779835,15.256276675195686,1.4852921380539956,20.22725858676771,81.04478048937415,494.29514441747557,0.028801384130459504,2026-02-25 13:10:01.815,FF-WASH,Washing,false,None,cd613e30-d8f1-6adf-91b7-584a2265b1f5
19.946881903081163,16.34657534653411,1.663304044710892,19.40325819814763,81.5697961831401,509.4736888775778,0.016652287063606663,2026-02-25 13:10:01.816,FF-WASH,Washing,false,None,cd613e30-d8f1-6adf-91b7-584a2265b1f5
20.355810162057566,14.87570967267973,1.7157134735368738,19.481031347928624,78.75988794645224,479.0827338815287,0.06472728509872108,2026-02-25 13:10:01.817,FF-WASH,Washing,false,None,cd613e30-d8f1-6adf-91b7-584a2265b1f5


### Step 4: Ingest data via Zerobus (single batch)

Using the **synchronous** SDK, each record is sent as a JSON dictionary.
The `wait_for_ack()` call ensures durability before moving to the next record.

In [0]:
def ingest_dataframe(df, sdk, table_props, options, client_id, client_secret, label="Batch"):
    """Ingest a pandas DataFrame into Delta via Zerobus sync SDK + JSON."""
    stream = sdk.create_stream(client_id, client_secret, table_props, options)
    total = len(df)
    start = time.time()
    successful = 0
    failed = 0

    for i in range(total):
        row = df.iloc[i]
        try:
            record = {
                "oil_temperature": float(row["oil_temperature"]),
                "water_temperature": float(row["water_temperature"]),
                "belt_speed": float(row["belt_speed"]),
                "freezer_temperature": float(row["freezer_temperature"]),
                "moisture_content": float(row["moisture_content"]),
                "product_weight": float(row["product_weight"]),
                "component_yield_output": float(row["component_yield_output"]),
                "timestamp": str(row["timestamp"]),
                "component_id": str(row["component_id"]),
                "stage_id": str(row["stage_id"]),
                "damaged_component": bool(row["damaged_component"]),
                "abnormal_sensor": str(row["abnormal_sensor"]) if row["abnormal_sensor"] is not None else "",
                "line_id": str(row["line_id"]),
            }
            ack = stream.ingest_record(record)
            ack.wait_for_ack()
            successful += 1
        except Exception as e:
            failed += 1
            if failed <= 3:
                print(f"  [WARN] Record {i} failed: {e}")

        # Progress update every 10%
        if (i + 1) % max(1, total // 10) == 0:
            pct = int((i + 1) / total * 100)
            print(f"  {label}: {pct}% ({i + 1}/{total})")

    stream.flush()
    stream.close()

    elapsed = time.time() - start
    throughput = successful / elapsed if elapsed > 0 else 0
    print(f"  {label} complete: {successful} records in {elapsed:.1f}s ({throughput:.0f} rec/s)")
    if failed > 0:
        print(f"  {label} failures: {failed}")
    return successful, failed

In [0]:
# Ingest the generated batch
print(f"Ingesting {len(batch_df_lines)} records via Zerobus...\n")
ok, fail = ingest_dataframe(
    batch_df_lines, sdk, table_props, options, CLIENT_ID, CLIENT_SECRET,
    label="Single Batch"
)
print(f"\nDone. Successful: {ok}, Failed: {fail}")

Ingesting 16000 records via Zerobus...

2026-02-25T12:16:22.353608Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=0927d50a-846e-416c-a835-02ed392aa9b6
2026-02-25T12:16:22.353637Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=0927d50a-846e-416c-a835-02ed392aa9b6
2026-02-25T12:16:22.353653Z  INFO databricks_zerobus_ingest_sdk: Successfully created new ephemeral stream stream_id=0927d50a-846e-416c-a835-02ed392aa9b6
  Single Batch: 10% (1600/16000)
  Single Batch: 20% (3200/16000)
2026-02-25T12:31:26.718394Z  INFO databricks_zerobus_ingest_sdk: Server will close the stream in 10000ms. Entering graceful close period (waiting up to 10000ms for in-flight acks).
2026-02-25T12:31:26.792555Z  INFO databricks_zerobus_ingest_sdk: All in-flight records acknowledged during graceful close. Triggering recovery.
2026-02-25T12:31:26.792594Z  INFO databricks_zerobus_ingest_sdk: Receiver task completed successfully
2026-02-25T12:31:27.247290Z  INFO

In [0]:
# Verify data in destination table
display(spark.table(BRONZE_TABLE).count())

67157

### Step 5: Parallel multi-line ingestion

Ingest data from each production line in parallel using threads.
Each thread creates its own Zerobus stream and ingests independently.

In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from line_data_generator import generate_line

line_sample_size = 1000

# Show per-line estimates
_, _, line_num_rows, est_line_table_size = table_size_estimator(
    machines_per_line, num_components, line_sample_size
)
for i in range(len(line_num_rows)):
    print(f"Line {i+1}: {line_num_rows[i]} rows ({est_line_table_size[i]:.2f} MB)")

Line 1: 6000 rows (2.86 MB)
Line 2: 5000 rows (2.38 MB)
Line 3: 5000 rows (2.38 MB)


In [0]:
def worker(line_number):
    """Thread worker: generate + ingest data for one production line."""
    line_name = ["French Fries", "Hash Browns", "Wedges"][line_number]
    print(f"Worker {line_number}: Starting Line {line_number+1} ({line_name})...")

    current_time = time.time()
    batch_df_line = generate_line(line_number, equipment_mapping, line_sample_size, current_time)

    ok, fail = ingest_dataframe(
        batch_df_line, sdk, table_props, options, CLIENT_ID, CLIENT_SECRET,
        label=f"Line {line_number+1} ({line_name})"
    )
    return line_number, ok, fail


print(f"Starting parallel ingestion for {num_lines} production lines...\n")
start_all = time.time()

with ThreadPoolExecutor(max_workers=num_lines) as executor:
    futures = [executor.submit(worker, i) for i in range(num_lines)]
    for future in as_completed(futures):
        line_num, ok, fail = future.result()
        print(f"  Line {line_num+1} finished: {ok} OK, {fail} failed")

print(f"\nAll lines done in {time.time() - start_all:.1f}s")

Starting parallel ingestion for 3 production lines...

Worker 0: Starting Line 1 (French Fries)...
Worker 1: Starting Line 2 (Hash Browns)...
Worker 2: Starting Line 3 (Wedges)...
2026-02-25T13:10:16.091478Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=3b19a5fa-1d77-414a-8fbd-bd54a804d97c
2026-02-25T13:10:16.091505Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=3b19a5fa-1d77-414a-8fbd-bd54a804d97c
2026-02-25T13:10:16.091558Z  INFO databricks_zerobus_ingest_sdk: Successfully created new ephemeral stream stream_id=3b19a5fa-1d77-414a-8fbd-bd54a804d97c
2026-02-25T13:10:16.091833Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=09360a31-3fff-4725-b453-5782014c7200
2026-02-25T13:10:16.091850Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=09360a31-3fff-4725-b453-5782014c7200
2026-02-25T13:10:16.091864Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=d77

In [0]:
# Verify data in destination table
display(spark.table(BRONZE_TABLE).count())

83146

### Step 6: Continuous data ingestion simulation

Simulate ongoing factory telemetry by running multiple batches with pauses
between them to avoid overlapping timestamps.

In [0]:
# Batch configuration
continuous_sample_size = 10000   # rows per component per batch
batch_count = 10                 # number of batches

tot_num_rows, est_table_size, _, _ = table_size_estimator(
    machines_per_line, num_components, continuous_sample_size
)
print(f"Rows per batch: {tot_num_rows}")
print(f"Size per batch: {est_table_size:.2f} MB")
print(f"Total rows ({batch_count} batches): {tot_num_rows * batch_count}")

Rows per batch: 160000
Size per batch: 76.29 MB
Total rows (10 batches): 1600000


In [0]:
run_simulation = True  # change to True to run

if run_simulation:
    stream = sdk.create_stream(CLIENT_ID, CLIENT_SECRET, table_props, options)
    min_batch_wait = continuous_sample_size * 0.001
    batch_time = 0

    for batch_num in range(batch_count):
        current_time = time.time()

        if batch_num > 0 and current_time <= batch_time + min_batch_wait:
            wait = int(batch_time + min_batch_wait - current_time + 10)
            print(f"Pausing {wait}s to avoid overlapping timestamps...")
            time.sleep(wait)

        print(f"\n--- Batch {batch_num + 1} / {batch_count} ---")
        batch_time = time.time()
        batch_df = generate_all_lines(equipment_mapping, continuous_sample_size, batch_time)

        total = len(batch_df)
        start = time.time()
        successful = 0

        for i in range(total):
            row = batch_df.iloc[i]
            record = {
                "oil_temperature": float(row["oil_temperature"]),
                "water_temperature": float(row["water_temperature"]),
                "belt_speed": float(row["belt_speed"]),
                "freezer_temperature": float(row["freezer_temperature"]),
                "moisture_content": float(row["moisture_content"]),
                "product_weight": float(row["product_weight"]),
                "component_yield_output": float(row["component_yield_output"]),
                "timestamp": str(row["timestamp"]),
                "component_id": str(row["component_id"]),
                "stage_id": str(row["stage_id"]),
                "damaged_component": bool(row["damaged_component"]),
                "abnormal_sensor": str(row["abnormal_sensor"]) if row["abnormal_sensor"] is not None else "",
                "line_id": str(row["line_id"]),
            }
            ack = stream.ingest_record(record)
            ack.wait_for_ack()
            successful += 1

            if (i + 1) % max(1, total // 5) == 0:
                print(f"  {int((i+1)/total*100)}% ({i+1}/{total})")

        stream.flush()
        elapsed = time.time() - start
        print(f"  Batch {batch_num+1}: {successful} rows in {elapsed:.1f}s")

    stream.close()
    print(f"\nContinuous simulation complete ({batch_count} batches)")

2026-02-25T13:45:29.862030Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=6a666ab7-16a0-4ab1-8795-53cfe9fb0c69

--- Batch 1 / 10 ---
2026-02-25T13:45:29.862061Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=6a666ab7-16a0-4ab1-8795-53cfe9fb0c69
2026-02-25T13:45:29.862078Z  INFO databricks_zerobus_ingest_sdk: Successfully created new ephemeral stream stream_id=6a666ab7-16a0-4ab1-8795-53cfe9fb0c69
2026-02-25T14:00:32.702258Z  INFO databricks_zerobus_ingest_sdk: Server will close the stream in 10000ms. Entering graceful close period (waiting up to 10000ms for in-flight acks).
2026-02-25T14:00:32.807616Z  INFO databricks_zerobus_ingest_sdk: All in-flight records acknowledged during graceful close. Triggering recovery.
2026-02-25T14:00:32.807657Z  INFO databricks_zerobus_ingest_sdk: Receiver task completed successfully
2026-02-25T14:00:33.388214Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=528d0108-399c-

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:717)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# Final table count
count = spark.table(BRONZE_TABLE).count()
print(f"\n{BRONZE_TABLE}: {count:,} total rows")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:717)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can